# AI CSPM Results Analysis: Context-Aware Cloud Auditor

## Мета дослідження
Цей інтерактивний зошит є головним аналітичним дашбордом для оцінки того, **чи здатні великі мовні моделі (LLM) ефективно виконувати роль контекстно-залежного хмарного аудитора**.

Традиційні інструменти Cloud Security Posture Management (CSPM), такі як AWS Security Hub та Prowler, часто генерують значну кількість False Positives, оскільки вони не розуміють бізнес-контексту, компенсаційних заходів (Compensating Controls) чи архітектурних рішень. Мета цього дослідження — перевірити, чи може ШІ аналізувати результати CSPM-сканувань разом із файлами конфігурації (Terraform) та архітектурними діаграмами для автоматичного відсіювання помилкових спрацьовувань та обґрунтованого коригування рівня ризику згідно зі стандартом NIST 800-53 Rev 5.

## Оцінювані метрики
У цьому звіті обчислюються та візуалізуються наступні ключові показники для інструментів **Prowler** та **AWS Security Hub**:

1. **Adjustment Rate (%)**: Відсоток знахідок CSPM, для яких LLM змінила критичність або позначила їх як False Positive.
2. **Decision Consistency (%)**: Стабільність рішень моделі. Показує, наскільки часто модель для однієї і тієї ж знахідки (в різних запусках) повертає ідентичну оцінку: ту саму фінальну критичність, той самий статус False Positive та ту саму причину виправлення (Adjustment Category).
3. **Розподіл категорій (Adjustment Categories)**: Аналіз того, *чому* модель змінює знахідки (наприклад, `COMPENSATING_CONTROL`, `ISOLATED_ENVIRONMENT`, `TOOL_INACCURACY`, `BUSINESS_REQUIREMENT` тощо).
4. **Ефективність та вартість**: Залежність між розміром контексту (вхідні токени), згенерованим виводом (вихідні токени) та швидкістю прийняття рішень (Latency) кожною моделлю.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

# 1. Load Data
metrics_path = '../output/analysis/metrics_history.csv'
findings_path = '../output/analysis/findings_history.csv'

if os.path.exists(metrics_path) and os.path.exists(findings_path):
    metrics_df = pd.read_csv(metrics_path)
    findings_df = pd.read_csv(findings_path)
    metrics_df = metrics_df.fillna(0)
    findings_df = findings_df.fillna("")
else:
    print("CSV files not found.")
    metrics_df = pd.DataFrame()
    findings_df = pd.DataFrame()

In [ ]:
if not metrics_df.empty and 'error_type' in metrics_df.columns:
    # 1. Prepare data for stacked bar chart
    status_df = metrics_df.groupby(['model', 'error_type']).size().unstack(fill_value=0)
    
    # Rename 'NONE' to 'OK'
    if 'NONE' in status_df.columns:
        status_df.rename(columns={'NONE': 'OK'}, inplace=True)
        
    # Ensure all expected columns exist
    for col in ['OK', 'ZERO_ADJUSTED', 'ALL_ADJUSTED', 'API_ERROR', 'TIMEOUT']:
        if col not in status_df.columns:
            status_df[col] = 0
            
    # Calculate Quality Rate (Success Rate)
    status_df['Total'] = status_df.sum(axis=1)
    status_df['Quality_Rate'] = (status_df['OK'] / status_df['Total']) * 100
    
    # Sort by Quality Rate descending
    status_df = status_df.sort_values(by='Quality_Rate', ascending=False)
    
    # Convert absolute counts to percentages for the stacked bar
    plot_cols = ['OK', 'ZERO_ADJUSTED', 'ALL_ADJUSTED', 'API_ERROR', 'TIMEOUT']
    percent_df = status_df[plot_cols].div(status_df['Total'], axis=0) * 100
    
    # Plotting
    colors = {
        'OK': 'mediumseagreen',
        'ZERO_ADJUSTED': 'orange',
        'ALL_ADJUSTED': 'gold',
        'API_ERROR': 'crimson',
        'TIMEOUT': 'darkred'
    }
    
    fig, ax = plt.subplots(figsize=(15, 6))
    
    percent_df.plot(kind='bar', stacked=True, color=[colors[c] for c in plot_cols], ax=ax, alpha=0.85)
    
    # Add Threshold Line
    threshold = 75.0
    ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Quality Threshold ({threshold}%)')
    
    ax.set_title("Model Stability & Edge Cases (Run Status Distribution)", fontsize=14, pad=15)
    ax.set_xlabel("Model", labelpad=10)
    ax.set_ylabel("Percentage of Runs (%)", labelpad=10)
    ax.set_ylim(0, 105)
    
    # Add percentage labels on the 'OK' bars for clarity
    for i, model in enumerate(percent_df.index):
        ok_pct = percent_df.loc[model, 'OK']
        if ok_pct > 5:  # Only label if there's enough space
            ax.text(i, ok_pct / 2, f'{ok_pct:.1f}%', ha='center', va='center', color='white', fontweight='bold')
    
    # Adjust legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], title="Status", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')
        
    fig.tight_layout()
    plt.show()

    # --- FILTERING ---
    # Keep only models that passed the threshold
    passed_models = status_df[status_df['Quality_Rate'] >= threshold].index.tolist()
    
    print(f"\n--- Quality Filter ---")
    print(f"Models passing >={threshold}% Quality Rate: {passed_models}")
    print(f"Models filtered out: {list(set(status_df.index) - set(passed_models))}\n")
    
    # Filter the global dataframes so all subsequent charts only show passed models
    metrics_df = metrics_df[metrics_df['model'].isin(passed_models)]
    if not findings_df.empty:
        findings_df = findings_df[findings_df['model'].isin(passed_models)]


In [2]:
def calculate_summary(m_df, f_df):
    summary = {
        "kpis": {},
        "model_comparison": [],
        "token_efficiency": [],
        "top_adjusted_controls": [],
        "category_distribution": []
    }
    
    if not m_df.empty:
        summary["kpis"]["total_runs"] = int(m_df["run_id"].nunique())
        summary["kpis"]["avg_adjustment_rate"] = float(m_df["adjustment_rate_percentage"].mean())
        summary["kpis"]["avg_prompt_tokens"] = float(m_df["prompt_tokens"].mean())
        summary["kpis"]["avg_completion_tokens"] = float(m_df["completion_tokens"].mean())
        summary["kpis"]["avg_latency"] = float(m_df["latency_seconds"].mean())
        
        model_group = m_df.groupby("model").agg({
            "adjustment_rate_percentage": "mean",
            "latency_seconds": "mean",
            "prompt_tokens": "mean",
            "completion_tokens": "mean"
        }).reset_index()
        
        for _, row in model_group.iterrows():
            summary["model_comparison"].append({
                "model": row["model"].replace("local/", "").split("/")[-1],
                "adjustment_rate": round(row["adjustment_rate_percentage"], 2),
                "latency": round(row["latency_seconds"], 2)
            })
            summary["token_efficiency"].append({
                "model": row["model"].replace("local/", "").split("/")[-1],
                "total_tokens": int(row["prompt_tokens"] + row["completion_tokens"]),
                "latency": round(row["latency_seconds"], 2)
            })
            
    if not f_df.empty:
        f_df = f_df.copy()
        f_df['decision_signature'] = f_df['adjusted_severity'].astype(str) + "_" + f_df['is_false_positive'].astype(str) + "_" + f_df['adjustment_category'].astype(str)
        consistency_df = f_df.groupby(['model', 'cspm_tool', 'finding_id', 'resource_id'])['decision_signature'].nunique().reset_index()
        consistency_df['is_consistent'] = consistency_df['decision_signature'] == 1
        overall_consistency = consistency_df['is_consistent'].mean() * 100
        summary["kpis"]["decision_consistency"] = float(round(overall_consistency, 2))
        
        def parse_controls(c):
            try:
                return json.loads(c) if isinstance(c, str) and c.startswith('[') else []
            except:
                return []
                
        f_df["nist_list"] = f_df["associated_nist_controls"].apply(parse_controls)
        exploded = f_df.explode("nist_list")
        is_fp = exploded["is_false_positive"].astype(str).str.lower() == "true"
        adjusted = exploded[(exploded["adjusted_severity"] != exploded["original_severity"]) | is_fp]
        
        if not adjusted.empty:
            if 'adjustment_category' not in adjusted.columns:
                adjusted['adjustment_category'] = 'NONE'
                
            top_controls = adjusted["nist_list"].value_counts().head(10).reset_index()
            top_controls.columns = ["control", "count"]
            
            for _, row in top_controls.iterrows():
                cats = adjusted[adjusted["nist_list"] == row["control"]]["adjustment_category"].value_counts()
                top_cat = cats.index[0] if not cats.empty and cats.index[0] != '' else "NONE"
                rtypes = adjusted[adjusted["nist_list"] == row["control"]]["resource_type"].value_counts()
                top_rtype = rtypes.index[0] if not rtypes.empty and rtypes.index[0] != '' else "unknown"
                summary["top_adjusted_controls"].append({
                    "control": row["control"],
                    "adjustments_count": int(row["count"]),
                    "common_category": top_cat,
                    "resource_type": top_rtype
                })
                
            cat_dist = adjusted["adjustment_category"].value_counts().reset_index()
            cat_dist.columns = ["category", "count"]
            for _, row in cat_dist.iterrows():
                if row["category"] and row["category"] != "NONE" and row["category"] != "":
                    summary["category_distribution"].append({
                        "name": row["category"],
                        "value": int(row["count"])
                    })
                    
    return summary


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Налаштування стилю графіків
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
def prep_consistency_data(m_df, f_df):
    if m_df.empty or f_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    f_df = f_df.copy()
    f_df['decision_signature'] = f_df['adjusted_severity'].astype(str) + "_" + f_df['is_false_positive'].astype(str) + "_" + f_df['adjustment_category'].astype(str)
    
    runs_per_model = m_df.groupby('model')['run_id'].nunique()
    
    consistency_df = f_df.groupby(['model', 'finding_id', 'resource_id'])['decision_signature'].nunique().reset_index()
    consistency_df['is_consistent'] = consistency_df['decision_signature'] == 1
    
    model_consistency = consistency_df.groupby('model')['is_consistent'].mean().reset_index()
    model_consistency.rename(columns={'is_consistent': 'consistency_rate'}, inplace=True)
    model_consistency['consistency_rate'] *= 100
    
    # Nullify consistency if only 1 run
    model_consistency['consistency_rate'] = model_consistency.apply(
        lambda row: row['consistency_rate'] if runs_per_model.get(row['model'], 0) > 1 else np.nan, 
        axis=1
    )
    
    # Sort by consistency rate descending
    model_consistency = model_consistency.sort_values(by='consistency_rate', ascending=False, na_position='last')
    
    run_adjustments = m_df[['run_id', 'model', 'adjustment_rate_percentage']].copy()
    run_adjustments['model'] = pd.Categorical(run_adjustments['model'], categories=model_consistency['model'], ordered=True)
    run_adjustments = run_adjustments.sort_values('model')
    
    return run_adjustments, model_consistency

if not metrics_df.empty:
    p_m = metrics_df[metrics_df['cspm_tool'].str.lower() == 'prowler']
    p_f = findings_df[findings_df['cspm_tool'].str.lower() == 'prowler']
    s_m = metrics_df[metrics_df['cspm_tool'].str.lower() == 'securityhub']
    s_f = findings_df[findings_df['cspm_tool'].str.lower() == 'securityhub']
    
    p_run, p_cons = prep_consistency_data(p_m, p_f)
    s_run, s_cons = prep_consistency_data(s_m, s_f)
    
    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(20, 6))
    
    # --- Prowler ---
    if not p_run.empty:
        sns.boxplot(data=p_run, x='model', y='adjustment_rate_percentage', ax=ax_p, hue='model', legend=False, palette='pastel', showfliers=False, width=0.4)
        sns.stripplot(data=p_run, x='model', y='adjustment_rate_percentage', ax=ax_p, palette='dark:black', alpha=0.5, jitter=False, hue='model', legend=False)
        ax_p.set_xlabel('Model')
        ax_p.set_ylabel('Adjustment Rate (%)', color='blue')
        ax_p.tick_params(axis='y', labelcolor='blue')
        ax_p.set_ylim(-5, 105)
        ax_p2 = ax_p.twinx()
        sns.scatterplot(data=p_cons, x='model', y='consistency_rate', ax=ax_p2, color='red', s=200, marker='D')
        
        # Draw lineplot only connecting non-NaN values
        valid_p_cons = p_cons.dropna(subset=['consistency_rate'])
        if not valid_p_cons.empty:
            sns.lineplot(data=valid_p_cons, x='model', y='consistency_rate', ax=ax_p2, color='red', alpha=0.5, sort=False)
            
        ax_p2.set_ylabel('Decision Consistency (%)', color='red')
        ax_p2.tick_params(axis='y', labelcolor='red')
        ax_p2.set_ylim(-5, 105)
        ax_p.set_title("Prowler: Adjustment Rate & Decision Consistency")
        for tick in ax_p.get_xticklabels():
            tick.set_rotation(45)
            tick.set_ha('right')
            
    # --- Security Hub ---
    if not s_run.empty:
        sns.boxplot(data=s_run, x='model', y='adjustment_rate_percentage', ax=ax_s, hue='model', legend=False, palette='pastel', showfliers=False, width=0.4)
        sns.stripplot(data=s_run, x='model', y='adjustment_rate_percentage', ax=ax_s, palette='dark:black', alpha=0.5, jitter=False, hue='model', legend=False)
        ax_s.set_xlabel('Model')
        ax_s.set_ylabel('Adjustment Rate (%)', color='blue')
        ax_s.tick_params(axis='y', labelcolor='blue')
        ax_s.set_ylim(-5, 105)
        ax_s2 = ax_s.twinx()
        sns.scatterplot(data=s_cons, x='model', y='consistency_rate', ax=ax_s2, color='red', s=200, marker='D', label='Decision Consistency')
        
        valid_s_cons = s_cons.dropna(subset=['consistency_rate'])
        if not valid_s_cons.empty:
            sns.lineplot(data=valid_s_cons, x='model', y='consistency_rate', ax=ax_s2, color='red', alpha=0.5, sort=False)
            
        ax_s2.set_ylabel('Decision Consistency (%)', color='red')
        ax_s2.tick_params(axis='y', labelcolor='red')
        ax_s2.set_ylim(-5, 105)
        ax_s.set_title("AWS Security Hub: Adjustment Rate & Decision Consistency")
        for tick in ax_s.get_xticklabels():
            tick.set_rotation(45)
            tick.set_ha('right')
        
        ax_s2.legend(bbox_to_anchor=(1.15, 1), loc='upper left')

    fig.tight_layout()
plt.show()

In [ ]:
def get_cat_percentages(f_df):
    if f_df.empty: return None
    f_df = f_df.copy()
    def parse_controls(c):
        try: return json.loads(c) if isinstance(c, str) and c.startswith('[') else []
        except: return []
    f_df["nist_list"] = f_df["associated_nist_controls"].apply(parse_controls)
    exploded = f_df.explode("nist_list")
    is_fp = exploded["is_false_positive"].astype(str).str.lower() == "true"
    adjusted = exploded[(exploded["adjusted_severity"] != exploded["original_severity"]) | is_fp]
    if adjusted.empty: return None
    if 'adjustment_category' not in adjusted.columns:
        adjusted['adjustment_category'] = 'NONE'
    adjusted['adjustment_category'] = adjusted['adjustment_category'].replace('', 'NONE')
    adjusted['adjustment_category'] = adjusted['adjustment_category'].fillna('NONE')
    cat_counts = adjusted.groupby(['model', 'adjustment_category']).size().unstack(fill_value=0)
    return cat_counts.div(cat_counts.sum(axis=1), axis=0) * 100

if not metrics_df.empty:
    p_cat = get_cat_percentages(p_f)
    s_cat = get_cat_percentages(s_f)
    
    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(20, 6))
    
    if p_cat is not None:
        p_cat.plot(kind='bar', stacked=True, color=sns.color_palette('husl', 12), ax=ax_p, legend=False)
        ax_p.set_title("Prowler: Розподіл категорій Adjustments (у %)")
        ax_p.set_xlabel("Model")
        ax_p.set_ylabel("Percentage (%)")
        for tick in ax_p.get_xticklabels():
            tick.set_rotation(45)
            tick.set_ha('right')
    else:
        ax_p.set_title("Prowler: No adjustments")
        
    if s_cat is not None:
        s_cat.plot(kind='bar', stacked=True, color=sns.color_palette('husl', 12), ax=ax_s, legend=False)
        ax_s.set_title("AWS Security Hub: Розподіл категорій Adjustments (у %)")
        ax_s.set_xlabel("Model")
        ax_s.set_ylabel("Percentage (%)")
        # Add legend only to the right plot, placed completely outside
        ax_s.legend(title="Adjustment Category", bbox_to_anchor=(1.05, 1), loc='upper left')
        for tick in ax_s.get_xticklabels():
            tick.set_rotation(45)
            tick.set_ha('right')
    else:
        ax_s.set_title("AWS Security Hub: No adjustments")

    fig.tight_layout()
    plt.show()


In [ ]:
if not metrics_df.empty:
    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(20, 6))
    
    p_df = p_m.copy()
    p_df['total_tokens'] = p_df['prompt_tokens'] + p_df['completion_tokens']
    s_df = s_m.copy()
    s_df['total_tokens'] = s_df['prompt_tokens'] + s_df['completion_tokens']
    
    if not p_df.empty:
        sns.scatterplot(data=p_df, x='total_tokens', y='latency_seconds', hue='model', size='adjustment_rate_percentage', sizes=(50, 400), alpha=0.7, palette='deep', ax=ax_p, legend=False)
        ax_p.set_title("Prowler: Токени vs Час виконання")
        ax_p.set_xlabel("Total Tokens")
        ax_p.set_ylabel("Latency (seconds)")
        
    if not s_df.empty:
        sns.scatterplot(data=s_df, x='total_tokens', y='latency_seconds', hue='model', size='adjustment_rate_percentage', sizes=(50, 400), alpha=0.7, palette='deep', ax=ax_s)
        ax_s.set_title("AWS Security Hub: Токени vs Час виконання")
        ax_s.set_xlabel("Total Tokens")
        ax_s.set_ylabel("Latency (seconds)")
        
        # Split legends
        h, l = ax_s.get_legend_handles_labels()
        if h:
            try:
                # Seaborn adds the column names as titles in the legend labels list
                split_idx = l.index('adjustment_rate_percentage')
                
                # Model legend
                l1, h1 = l[1:split_idx], h[1:split_idx]  # skip the 'model' title string at index 0
                leg1 = ax_s.legend(h1, l1, title="Model", bbox_to_anchor=(1.05, 1), loc='upper left')
                ax_s.add_artist(leg1)
                
                # Adjustment Rate legend
                l2, h2 = l[split_idx+1:], h[split_idx+1:]
                leg2 = ax_s.legend(h2, l2, title="Adjustment Rate (%)", bbox_to_anchor=(1.05, 0.4), loc='upper left')
            except ValueError:
                # Fallback if names differ
                ax_s.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    fig.tight_layout()
    plt.show()

In [ ]:
if not metrics_df.empty:
    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(20, 6))
    
    if not p_df.empty:
        sns.scatterplot(data=p_df, x='total_unique_nist_controls_failed', y='total_tokens', hue='model', s=150, alpha=0.8, palette='deep', ax=ax_p, legend=False)
        ax_p.set_title("Prowler: Токени vs Кількість правил (Token Efficiency)")
        ax_p.set_xlabel("Number of Failed Controls Processed")
        ax_p.set_ylabel("Total Tokens Spent")
        
    if not s_df.empty:
        sns.scatterplot(data=s_df, x='total_unique_nist_controls_failed', y='total_tokens', hue='model', s=150, alpha=0.8, palette='deep', ax=ax_s)
        ax_s.set_title("AWS Security Hub: Токени vs Кількість правил (Token Efficiency)")
        ax_s.set_xlabel("Number of Failed Controls Processed")
        ax_s.set_ylabel("Total Tokens Spent")
        ax_s.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    fig.tight_layout()
    plt.show()

In [3]:
# Generate for Prowler and SecurityHub
final_export = {
    "all": calculate_summary(metrics_df, findings_df)
}

if not metrics_df.empty:
    prowler_metrics = metrics_df[metrics_df["cspm_tool"].str.lower() == "prowler"]
    sh_metrics = metrics_df[metrics_df["cspm_tool"].str.lower() == "securityhub"]
    
    prowler_findings = findings_df[findings_df["cspm_tool"].str.lower() == "prowler"]
    sh_findings = findings_df[findings_df["cspm_tool"].str.lower() == "securityhub"]
    
    final_export["prowler"] = calculate_summary(prowler_metrics, prowler_findings)
    final_export["securityhub"] = calculate_summary(sh_metrics, sh_findings)

# Export
presentation_dir = "../presentation/public"
os.makedirs(presentation_dir, exist_ok=True)
out_path = os.path.join(presentation_dir, "summary.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_export, f, indent=2, ensure_ascii=False)

print(f"Аналіз завершено. Дані збережено до {out_path}")

Аналіз завершено. Дані збережено до ../presentation/public/summary.json
